# EnerGIS Framework - Runner

Haupteinstiegspunkt für Optimierungsläufe mit dem EnerGIS Planning Framework.

## Übersicht

Dieses Notebook führt einen vollständigen Optimierungslauf durch:
- **Perfect Forecast (PF)**: Optimale Dimensionierung über den gesamten Zeitraum
- **Rolling Horizon (RH)**: Operative Planung mit rollendem Horizont
- **PF → RH**: Kombinierter Workflow mit Design-Fixierung

## Quick Start

1. Alle Zellen mit **Run All** ausführen
2. Bei Bedarf Config-Pfade in Zelle 2 anpassen
3. Ergebnisse werden in `exports/` gespeichert

---

## 1. Setup & Imports

In [6]:
# Auto-Setup: Projekt-Root finden und zum Path hinzufügen
from pathlib import Path
import sys
import os

def find_project_root(start: Path) -> Path:
    """Findet das Projekt-Root-Verzeichnis."""
    for candidate in [start] + list(start.parents):
        if (candidate / '.git').exists() and (candidate / 'energis').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Projekt-Root: {PROJECT_ROOT}")

✅ Projekt-Root: c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat


In [7]:
# Imports
import warnings
from datetime import datetime

from energis.run import rolling_horizon as rh
from energis.run import orchestrator

warnings.filterwarnings('ignore')
print("✅ Imports erfolgreich")

✅ Imports erfolgreich


## 2. Konfiguration

Die Konfiguration erfolgt über YAML-Dateien, die in der angegebenen Reihenfolge gemerged werden.
Spätere Dateien überschreiben frühere Einträge.

### Standard-Konfiguration:
- `base.yaml` - Basis-Einstellungen (Solver, Zeitschritt, etc.)
- `tech_catalog.yaml` - Technologie-Katalog (Komponenten-Definitionen)
- `default.site.yaml` - Standort-Daten (Input-Daten, Zeitzone, etc.)
- `baseline.system.yaml` - System-Topologie (Komponenten, Kapazitäten)
- `pf_then_rh.workflow.scenario.yaml` - Szenario (Run-Mode, RH-Parameter)

Passe die Config-Pfade nach Bedarf an!

In [8]:
# Konfigurationsdateien
CONFIG_PATHS = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/test_1week.scenario.yaml',
]

# Optional: Overrides für spezifische Parameter
# Beispiele:
# - Run-Mode ändern: {'scenario': {'run_mode': 'PF_ONLY'}}
# - Solver ändern: {'run': {'solver': 'glpk'}}
# - RH-Parameter: {'scenario': {'rolling_horizon': {'heat_horizon_hours': 72}}}
OVERRIDES = None

# Config-Dateien prüfen
print("📋 Konfigurationsdateien:")
all_exist = True
for cfg_path in CONFIG_PATHS:
    full_path = PROJECT_ROOT / cfg_path
    exists = full_path.exists()
    symbol = '✅' if exists else '❌'
    print(f"  {symbol} {cfg_path}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

📋 Konfigurationsdateien:
  ✅ configs/base.yaml
  ✅ configs/tech_catalog.yaml
  ✅ configs/sites/default.site.yaml
  ✅ configs/systems/baseline.system.yaml
  ✅ configs/scenarios/test_1week.scenario.yaml

✅ Konfiguration OK


## 3. Workflow ausführen

Der Workflow führt die Optimierung gemäß der konfigurierten Run-Mode aus:
- **PF_ONLY**: Nur Perfect Forecast
- **RH_ONLY**: Nur Rolling Horizon
- **PF_THEN_RH**: PF für Dimensionierung, dann RH mit fixiertem Design

In [9]:
%%time
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False

🚀 STARTE OPTIMIERUNG
Start: 2025-11-18 12:24:58

[LOAD] Import_Data.xlsx → 35037 Schritte von 2023-01-01 00:00:00 bis 2023-12-31 23:00:00
[SCENARIO] Zeitraum 2023-01-01 00:00:00 → 2023-01-08 00:00:00 (673 Schritte)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1

✅ OPTIMIERUNG ERFOLGREICH

📊 Workflow: PF
CPU times: total: 50min 27s
Wall time: 13min 23s


## 4. Ergebnisse

Zeigt die wichtigsten Kennzahlen aus dem Optimierungslauf.

In [10]:
if optimization_success and workflow:
    print("\n" + "="*70)
    print("📊 ERGEBNISSE")
    print("="*70)
    
    # Perfect Forecast Ergebnisse
    if workflow.pf_result:
        print("\n🎯 Perfect Forecast (PF):")
        print(f"  Zeitschritte:  {len(workflow.pf_result.table)}")
        
        if workflow.pf_result.costs:
            obj_value = workflow.pf_result.costs.get('objective.OBJ_value_EUR')
            if obj_value is not None:
                print(f"  Gesamtkosten:  {obj_value:,.0f} EUR")
            
            peak_power = workflow.pf_result.costs.get('P_buy_peak_MW')
            if peak_power is not None:
                print(f"  Peak-Leistung: {peak_power:.2f} MW")
    
    # Rolling Horizon Ergebnisse
    if workflow.rh_result:
        print("\n🔄 Rolling Horizon (RH):")
        print(f"  Fenster:       {len(workflow.rh_result.windows)}")
        print(f"  Zeitschritte:  {len(workflow.rh_result.table)}")
        
        if workflow.rh_result.costs:
            obj_value = workflow.rh_result.costs.get('objective.OBJ_value_EUR')
            if obj_value is not None:
                print(f"  Gesamtkosten:  {obj_value:,.0f} EUR")
    
    # Design
    if workflow.design:
        print("\n🏭 Anlagen-Design:")
        
        if workflow.design.heat_pumps:
            print("  Wärmepumpen:")
            for hp_id, capacity in sorted(workflow.design.heat_pumps.items()):
                print(f"    {hp_id}: {capacity:.2f} MW")
        
        if workflow.design.storage:
            print(f"  Speicher:      {workflow.design.storage:.2f} MWh")
        
        if workflow.design.generators:
            print("  Generatoren:")
            for gen_id, capacity in sorted(workflow.design.generators.items()):
                print(f"    {gen_id}: {capacity:.2f} MW")
    
    print("\n" + "="*70)
else:
    print("⚠️  Keine Ergebnisse verfügbar")


📊 ERGEBNISSE

🎯 Perfect Forecast (PF):
  Zeitschritte:  673
  Gesamtkosten:  247,836 EUR

🏭 Anlagen-Design:
  Wärmepumpen:


TypeError: unsupported format string passed to dict.__format__

## 5. Export (Optional)

Vollständiger Export aller Ergebnisse als Excel/CSV/JSON in `exports/`.

In [14]:
 Uncomment to export results
if optimization_success:
    print("📦 Exportiere Ergebnisse...")
    export_meta = orchestrator.run_all(CONFIG_PATHS, overrides=OVERRIDES)
    
    print(f"\n✅ Export abgeschlossen:")
    print(f"  Verzeichnis: {export_meta['outdir']}")
    print(f"  Excel:       {export_meta.get('scenario_xlsx')}")
    print(f"  Design-JSON: {export_meta.get('pf_design_json')}")

IndentationError: unexpected indent (587058154.py, line 1)

## 6. Detaillierte Analyse (Optional)

Für detaillierte Analysen und Visualisierungen siehe:
- `scenario_studio.ipynb` - Interaktives Dashboard mit Plots und KPIs
- `synthetic_example.ipynb` - Beispiel mit synthetischen Daten
- `validation.ipynb` - Validierung gegen Referenz-Daten

---

## Weitere Informationen

- **Dokumentation**: `README.md`, `ARCHITECTURE_V2.md`
- **Methodologie**: `docs/methodology.md`
- **CLI-Nutzung**: `python -m energis.run.rolling_horizon --help`